In [1]:
import math
from html.parser import incomplete_charref

import dnnlpy
import dnnlpy.nn.functional as dF
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor
from torch.ao.nn.quantized import reference

print("PyTorch version: ", torch.__version__)


PyTorch version:  2.13.0+cpu


In [2]:
"""
nn.Conv2d()的输入,权重,输出
    Cin 决定一个卷积核需要观察多少个通道
    Cout 决定卷积层要学习多少组不同的特征提取方式
"""
conv = nn.Conv2d(in_channels=3, out_channels=8, kernel_size=(3, 5), stride=1, padding=(1, 2))
print('weight shape:', conv.weight.shape)
print('bias shape:', conv.bias.shape)

weight shape: torch.Size([8, 3, 3, 5])
bias shape: torch.Size([8])


In [3]:
"""
统一处理二元组参数
"""


def as_tuple(value: int | tuple[int, int]) -> tuple[int, int] | tuple[int, ...]:
    if isinstance(value, int):
        return value, value
    if len(value) != 2:
        raise AssertionError("excepted  in integer  a sequence of length 2")
    return tuple(map(int, value))


print('3->', as_tuple(3))
print("(3,5)->", as_tuple((3, 5)))


# 计算输出维度
def conv_output_size(input_size: int, kernel_size: int, padding: int, stride: int) -> int:
    output_size = (input_size + 2 * padding - kernel_size) // stride + 1
    if output_size < 0:
        raise RuntimeError("calculated output size too small")

    return output_size

3-> (3, 3)
(3,5)-> (3, 5)


In [4]:
"""
用4层循环实现多通道卷积
    1. batch中的每个样本
    2. 每个通道的输出
    3. 输出特征图中的每一行
    4. 输出特征图中的每一列
"""


def conv2d_v1(
        x: Tensor,
        weight: Tensor,
        bias: Tensor | None = None,
        stride: int | tuple[int, int] = 1,
        padding: int | tuple[int, int] = 0,
) -> Tensor:
    """

    :param x:
    :param weight:
    :param bias:
    :param stride:
    :param padding:
    :return:
    """
    if x.ndim != 4:
        raise AssertionError("Input must shape(B,C_in,H,W)")
    if weight.ndim != 4:
        raise AssertionError("Weight must shape(C_out,C_in,K_h,K_w)")
    if x.size(1) != weight.size(1):
        raise AssertionError("Input size must equal weight size")
    if bias is not None and weight.size(0) != bias.size(0):
        raise AssertionError("Bias must have same size")

    s_h, s_w = as_tuple(stride)
    p_h, p_w = as_tuple(padding)

    if s_h <= 0 or s_w <= 0:
        raise AssertionError("stride must be greater than 0")
    if p_h < 0 or p_w < 0:
        raise AssertionError("padding must be greater than 0")

    batch_size, in_channels, input_h, input_w = x.size()
    out_channels, _, k_h, k_w = weight.size()

    output_h = conv_output_size(input_h, k_h, p_h, s_h)
    output_w = conv_output_size(input_w, k_w, p_w, s_w)
    x_padded = F.pad(x, pad=(p_w, p_w, p_h, p_h))
    output = x.new_empty(batch_size, out_channels, output_h, output_w)

    for B in range(batch_size):
        for out_channel in range(out_channels):
            for i in range(output_h):
                row_start = i * s_h
                row_end = row_start + k_h
                for j in range(output_w):
                    col_start = j * s_w
                    col_end = col_start + k_w
                    window = x_padded[B, :, row_start:row_end, col_start:col_end]
                    value = torch.sum(window * weight[out_channel])

                    if bias is not None:
                        value = value + bias[out_channel]
                    output[B, out_channel, i, j] = value
    return output


x = torch.randn(2, 3, 5, 6)
weight = torch.randn(4, 3, 3, 2)
bias = torch.randn(4)

actual = conv2d_v1(x, weight, bias, stride=(2, 1), padding=(1, 0))

print("Input shape: ", x.shape)
print("Weight shape: ", weight.shape)
print("Output shape: ", actual.shape)



Input shape:  torch.Size([2, 3, 5, 6])
Weight shape:  torch.Size([4, 3, 3, 2])
Output shape:  torch.Size([2, 4, 3, 5])


In [5]:
"""
与PyTorch的F.conv2d对照
"""
expeted = F.conv2d(x, weight, bias, stride=(2, 1), padding=(1, 0))
# print(expeted)
# print(actual)
tets = (actual - expeted).abs()
max_err = (actual - expeted).abs().max()
print("maximun absolute error: ", max_err.item())

flag = torch.allclose(actual, expeted, atol=1e-5)
print("Is the hand-written  implementation  close  to F.conv2d.", flag)

maximun absolute error:  1.9073486328125e-06
Is the hand-written  implementation  close  to F.conv2d. True


In [7]:
"""
自动微分是否仍然有效
"""


def _copy(x: Tensor, mode: bool = True) -> Tensor:
    return x.detach().clone().requires_grad_(mode)


x_actual = torch.randn(1, 2, 4, 5, requires_grad=True)
weight_actual = torch.randn(3, 2, 3, 2, requires_grad=True)
bias_actual = torch.randn(3, requires_grad=True)

x_expected = _copy(x_actual)
weight_expected = _copy(weight_actual)
bias_expected = _copy(bias_actual)

actual = conv2d_v1(
    x_actual,
    weight_actual,
    bias_actual,
    stride=(2, 1),
    padding=(1, 0)
)

loss_actual = actual.square().mean()
loss_actual.backward()

expected = F.conv2d(
    x_expected,
    weight_expected,
    bias_expected,
    stride=(2, 1),
    padding=(1, 0)
)

loss_expected = expected.square().mean()
loss_expected.backward()

flag = torch.allclose(x_actual.grad, x_expected.grad, atol=1e-5)
print('Is Input  gradient  expected', flag)
flag = torch.allclose(weight_actual.grad, weight_expected.grad, atol=1e-5)
print('Is weight  gradient  expected', flag)
flag = torch.allclose(bias_actual.grad, bias_actual.grad, atol=1e-5)
print('Is bias  gradient  expected', flag)


Is Input  gradient  expected True
Is weight  gradient  expected True
Is bias  gradient  expected True


In [20]:
"""
把函数封装成Conv2d模块
"""


class Conv2d(nn.Module):
    def __init__(self,
                 in_channels: int,
                 out_channels: int,
                 kernel_size: tuple[int, int],
                 stride: tuple[int, int],
                 padding: tuple[int, int],
                 bias: bool = True):
        super().__init__()

        if in_channels < 0 or out_channels <= 0:
            raise AssertionError("in_channels and out_channels must be greater than 0")
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.kernel_size = as_tuple(kernel_size)
        self.stride = as_tuple(stride)
        self.padding = as_tuple(padding)

        t_h, t_w = self.kernel_size
        self.weight = nn.Parameter(torch.Tensor(out_channels, in_channels, t_h, t_w))
        if bias:
            self.bias = nn.Parameter(torch.empty(out_channels))
        else:
            self.register_parameter('bias', None)
        self.reset_parameters()

    def reset_parameters(self) -> None:
        nn.init.kaiming_uniform_(self.weight, a=math.sqrt(5))
        if self.bias is not None:
            fan_in = self.in_channels * math.prod(self.kernel_size)
            bound = 1 / math.sqrt(fan_in)
            nn.init.uniform_(self.weight, -bound, bound)

    def forward(self, x: Tensor) -> Tensor:
        return conv2d_v1(x, weight=self.weight, bias=self.bias, stride=self.stride, padding=self.padding)

    def extra_repr(self) -> tuple[str, str, str]:
        return (f'{self.in_channels},{self.out_channels}, ',
                f'kernel_size: {self.kernel_size}, stride: {self.stride}, ',
                f'padding: {self.padding}, bias: {self.bias is not None}')


custom_conv = Conv2d(in_channels=3, out_channels=8, kernel_size=3, stride=2, padding=1)

print("weight shape: ", custom_conv.weight.shape)
print("bias shape: ", custom_conv.bias.shape)
print("number of parameters: ", dnnlpy.count_params(custom_conv))



weight shape:  torch.Size([8, 3, 3, 3])
bias shape:  torch.Size([8])
number of parameters:  224


In [ ]:
"""
为什么这样初始化权重
"""



In [21]:
"""
与PyTorch的nn.Conv2d 对照
"""
custom_conv = Conv2d(
    in_channels=2,
    out_channels=3,
    kernel_size=(3, 3),
    stride=(1, 1),
    padding=(1, 1),
)
reference_conv = nn.Conv2d(
    in_channels=2,
    out_channels=3,
    kernel_size=(3, 3),
    stride=(1, 1),
    padding=(1, 1),
)

with torch.no_grad():
    reference_conv.weight.copy_(custom_conv.weight)
    reference_conv.bias.copy_(custom_conv.bias)

actual = torch.randn(2, 2, 6, 5, requires_grad=True)
expected = _copy(actual)

actual = custom_conv(actual)
expected = reference_conv(expected)

flag = torch.allclose(actual, expected, atol=1e-5)
print("Is the hand-written  implementation  close  to F.conv2d.", flag)

loss_actual = actual.square().mean()
loss_expected = expected.square().mean()

loss_actual.backward()
loss_expected.backward()

flag = torch.allclose(actual, expected, atol=1e-5)
print("Is input   gradient  expected", flag)

flag = torch.allclose(custom_conv.weight, reference_conv.weight, atol=1e-5)
print("Is weight  gradient  expected", flag)

flag = torch.allclose(custom_conv.bias.grad, reference_conv.bias.grad, atol=1e-5)
print("Is bias  gradient  expected", flag)

Is the hand-written  implementation  close  to F.conv2d. True
Is input   gradient  expected True
Is weight  gradient  expected True
Is bias  gradient  expected True


In [22]:
"""
为什么真正的Conv2d不会这样写
"""


def conv2d_v2(
        x: Tensor,
        weight: Tensor,
        bias: Tensor | None,
        stride: int | tuple[int, int] = 1,
        padding: int | tuple[int, int] = 0,
) -> Tensor:
    if x.ndim != 4:
        raise AssertionError("Expected 4D tensor, got {}".format(x.ndim))
    if weight.ndim != 4:
        raise AssertionError("Expected 4D tensor, got {}".format(weight.ndim))
    if x.size(1) != weight.size(1):
        raise AssertionError("Input channels must  match  weight  channels ")
    if bias is not None and weight.size(0) != bias.size(0):
        raise AssertionError("Bias  must  match  bias (C_out) ")

    stride = as_tuple(stride)
    padding = as_tuple(padding)

    batch_size, in_channels, input_h, input_w = x.shape
    out_channels, _, k_h, k_w = weight.shape

    output_h = conv_output_size(input_h, k_h, padding[0], stride[0])
    output_w = conv_output_size(input_w, k_w, padding[1], stride[1])

    patches = dF.unfold(
        x,
        kernel_size=(k_h, k_w),
        padding=padding,
        stride=stride,
    )

    weight = weight.reshape(out_channels, -1)
    output = weight @ patches
    if bias is not None:
        output = output + bias.reshape(1, -1, 1)
    return output.reshape(batch_size, out_channels, output_h, output_w)


x = torch.randn(2, 3, 7, 6)
weight = torch.randn(5, 3, 3, 2)
bias = torch.randn(5)

actual = conv2d_v2(x, weight, bias, stride=(2, 1), padding=(1, 0))
excepted = F.conv2d(x, weight, bias, stride=(2, 1), padding=(1, 0))

flag = torch.allclose(actual, excepted, atol=1e-5)
print("Is the hand-written  implementation  close  to F.conv2d.", flag)




Is the hand-written  implementation  close  to F.conv2d. True
